# MoCo: Momentum Contrast

_Learning from Unlabeled Data (semi- & self-supervised)_

**Get SimCLR's many negatives for free with a memory queue and a slowly-moving key encoder.**

Contrastive learning [mod-contrastive] makes two augmented views of the same image agree, while pushing them away from views of other images (the negatives). The more negatives you contrast against, the sharper the features. SimCLR [unl-simclr] gets negatives from the current mini-batch, so it needs a huge batch — which means many GPUs.

       MoCo (Momentum Contrast) gets the same many negatives cheaply with two ideas:

---

This notebook is a practice scaffold for the **MoCo: Momentum Contrast** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoCo(nn.Module):
    """Momentum Contrast (He et al. 2019). v2: pass an MLP head as encoder."""
    def __init__(self, encoder_q, encoder_k, dim=128, K=65536, m=0.999, T=0.07):
        super().__init__()
        self.K, self.m, self.T = K, m, T
        self.encoder_q = encoder_q          # trained by gradient descent
        self.encoder_k = encoder_k          # momentum (EMA) copy, no grads

        # key encoder starts as an exact copy of the query encoder
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.copy_(pq.data)
            pk.requires_grad = False        # never updated by backprop

        # the dictionary queue: dim x K column-normalized key vectors
        self.register_buffer("queue", F.normalize(torch.randn(dim, K), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def _momentum_update(self):
        # theta_k <- m * theta_k + (1 - m) * theta_q
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.mul_(self.m).add_(pq.data, alpha=1.0 - self.m)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        batch = keys.shape[0]
        ptr = int(self.queue_ptr)
        # overwrite the oldest 'batch' columns with the new keys (wrap allowed for divisor sizes)
        self.queue[:, ptr:ptr + batch] = keys.T
        self.queue_ptr[0] = (ptr + batch) % self.K

    def forward(self, im_q, im_k):
        # 1) query features (gradients flow here)
        q = F.normalize(self.encoder_q(im_q), dim=1)        # (N, dim)

        # 2) key features from the momentum encoder (no gradients)
        with torch.no_grad():
            self._momentum_update()
            # NOTE: real MoCo shuffles im_k across GPUs here (Shuffling BN) to
            # stop BatchNorm statistics from leaking the positive's identity.
            k = F.normalize(self.encoder_k(im_k), dim=1)    # (N, dim)

        # 3) InfoNCE logits: positive is column 0, negatives are the queue
        l_pos = torch.einsum("nc,nc->n", q, k).unsqueeze(1)            # (N, 1)
        l_neg = torch.einsum("nc,ck->nk", q, self.queue.clone())      # (N, K)
        logits = torch.cat([l_pos, l_neg], dim=1) / self.T            # (N, 1+K)

        # the correct class is index 0 (the positive) for every query
        labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, labels)               # == InfoNCE

        # 4) push the fresh keys into the queue, drop the oldest
        self._dequeue_and_enqueue(k)
        return loss

# --- one optimization step ---
# loss = model(im_q, im_k)          # im_q, im_k = two augmented views of a batch
# optimizer.zero_grad(); loss.backward(); optimizer.step()

## Visualize the data & results

_On real digit embeddings, how does the chance of correctly matching an augmented view to its original change as the number of negatives grows?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# real bundled dataset: 1797 handwritten digits, 64 pixels each
X = load_digits().data.astype(float) / 16.0
# a small encoder proxy: PCA to a 16-D embedding (so noisy views are confusable)
emb = PCA(n_components=16, random_state=0).fit_transform(X)
n = emb.shape[0]

def topk_acc(N, sigma=1.5, trials=600, seed=0):
    """Match an augmented view of one image to its original among N candidates."""
    rng = np.random.RandomState(seed)
    correct = 0
    for _ in range(trials):
        idx = rng.choice(n, N, replace=False)          # N originals = the 'keys'
        gallery = normalize(emb[idx])                  # L2-normalize -> cosine
        pos = rng.randint(N)                           # which one is the positive
        q = emb[idx[pos]] + rng.normal(0, sigma, emb.shape[1])  # 'augmented' query
        q = q / (np.linalg.norm(q) + 1e-9)
        sims = gallery @ q                             # cosine similarity to each key
        correct += (np.argmax(sims) == pos)            # top-1 retrieval hit
    return correct / trials

Ns = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
acc = [topk_acc(N, seed=N) for N in Ns]
chance = [1.0 / N for N in Ns]

xs = np.log2(Ns)
plt.plot(xs, acc, "-o", c="#4ea1ff", label="MoCo-style retrieval (cosine match)")
plt.plot(xs, chance, "--", c="#9aa7b4", label="random-guess baseline (1/N)")
plt.title("Contrastive retrieval: top-1 match accuracy vs number of negatives (load_digits)")
plt.xlabel("number of negatives N (log scale of 2^x)")
plt.ylabel("top-1 match accuracy")
plt.legend()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** MoCo uses temperature $\tau = 0.2$. A query scores $q\cdot k_+ = 0.8$ against its positive and $q\cdot k_1 = 0.4$ against a single queued negative ($K = 1$). What probability does the loss assign to the positive, and what is the loss?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Divide both similarities by $\tau = 0.2$. — _The temperature scales every score before the softmax; this is the "sharpening" knob._
- Exponentiate each scaled score. — _InfoNCE is a softmax, so each candidate's weight is $\exp(\text{score})$._
- Form the probability of the positive as its weight over the sum of all weights, then take $-\log$. — _The loss is the negative log-probability of picking the correct (positive) key._

**Answer:** Scaled: $0.8/0.2 = 4.0$ and $0.4/0.2 = 2.0$. Exponentiate: $e^{4.0} \approx 54.6$, $e^{2.0} \approx 7.39$. Probability of the positive: $\frac{54.6}{54.6 + 7.39} = \frac{54.6}{61.99} \approx 0.881$. Loss $= -\log(0.881) \approx 0.127$. With only one negative the loss is small; a full queue of thousands of negatives would push it higher and give a stronger learning signal.

</details>

**Problem 2.** The key encoder uses momentum $m = 0.999$. In plain English, how far does $\theta_k$ move toward $\theta_q$ in one step, and why must $m$ be this close to 1?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the update $\theta_k \leftarrow m\,\theta_k + (1-m)\,\theta_q$ and find the fraction of the gap closed. — _The new weights are a blend; $(1-m)$ is the share taken from the query encoder._
- Connect a slow-moving $f_k$ to the consistency of keys made many batches ago. — _The queue compares keys made at different times, so they must come from a near-identical encoder._

**Answer:** $1 - m = 0.001$, so the key encoder moves just $0.1\%$ of the way toward the query encoder each step — it barely changes. It must be this slow because the queue holds keys produced hundreds of steps ago; if $f_k$ jumped each step, those old keys would come from a very different network and comparing them to today's query would be meaningless. A high $m$ keeps the whole queue consistent, which is the trick that lets MoCo use a huge bank of negatives.

</details>

**Problem 3.** Why can MoCo see 65{,}536 negatives with a batch size of only 256, while SimCLR would need a batch of 65{,}536 to see the same number?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall where each method's negatives come from. — _SimCLR's negatives are the other images in the current batch; MoCo's are the queue._
- Note that the queue persists across steps and is decoupled from the forward pass. — _Queued keys were computed in earlier steps and are just stored vectors, not re-encoded now._

**Answer:** SimCLR draws negatives from the current mini-batch, so the only way to get $N$ negatives is a batch of about $N$ — which is why it needs thousands-wide batches and many GPUs. MoCo stores keys from past batches in a queue; those vectors are already computed and just sit in memory. Each step adds 256 fresh keys and drops the 256 oldest, but the queue can hold 65{,}536 of them. So the number of negatives is set by the queue length, not the batch size, and MoCo gets SimCLR-scale negatives on modest hardware.

</details>